In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import pyautogui
import time
import pyscreenshot as ImageGrab
import pytesseract
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image, ImageOps, ImageEnhance
import csv
import cv2
import os
import re
pytesseract.pytesseract.tesseract_cmd = '/opt/homebrew/bin/tesseract'

In [ ]:
url = "https://vr.vex.com"
driver = webdriver.Chrome()
driver.get(url)
time.sleep(0.5)
login = driver.find_element(By.XPATH, '//*[@id="app"]/div[10]/div/div/div[3]/div[2]/p/a')
login.click()
code = driver.find_element(By.XPATH, '//*[@id="app"]/div[10]/div/div/div[2]/div[1]/input')
code.send_keys('use your own team number here')
key = driver.find_element(By.XPATH, '//*[@id="app"]/div[10]/div/div/div[2]/div[2]/input')
key.send_keys('use your own key here')
submit = driver.find_element(By.XPATH, '//*[@id="app"]/div[10]/div/div/div[2]/button')
submit.click()
time.sleep(1)
close = driver.find_element(By.XPATH, '/html/body/div[10]/div/div[1]//a')
close.click()

In [4]:
start_button_pos = (545, 720)
res_button_pos = (545, 754)
submit_button_pos = (750, 713)
retry_button_pos = (836, 712)
retry_button_colour = (197, 58, 55)
python_interpreter = (808, 576)
intepreter_colour = (193, 142, 48)
score_x1 = 523
score_y1 = 452
score_x2 = 590
score_y2 = 466
screenshot_x1 = 738
screenshot_y1 = 616
screenshot_x2 = 894
screenshot_y2 = 636
bbox = (screenshot_x1, screenshot_y1, screenshot_x2, screenshot_y2)
ss_x1 = 536
ss_y1 = 447
ss_x2 = 587
ss_y2 = 466
bbox2 = (screenshot_x1, screenshot_y1, screenshot_x2, screenshot_y2)
bbox3 = (score_x1, score_y1, score_x2, score_y2)

In [ ]:
def extract_score():
    try:
        img = ImageGrab.grab(bbox)
        inverted_img = ImageOps.invert(img.convert('RGB'))
        img_np = np.array(inverted_img)
        img_np = np.where(img_np > 128, 255, 0)
        b_w = Image.fromarray(img_np.astype(np.uint8))
        
        text = pytesseract.image_to_string(
            b_w, 
            config='--psm 8 -c tessedit_char_whitelist=0123456789'
        )
        
        text = text.strip()
        
        if not text:
            print(f"Score: 0\n")
            return 0
        
        if text.isdigit():
            score = int(text)
        else:
            digits = ''.join(filter(str.isdigit, text))
            if digits:
                score = int(digits)
            else:
                print(f"Score: 0\n")
                return 0
        
        print(f"Score: {score}\n")
        return score
        
    except Exception as e:
        print(f"Score: 0\n")
        return 0

def extract_score2():
    img = ImageGrab.grab(bbox2)
    inverted_img = ImageOps.invert(img.convert('RGB'))
    img_np = np.array(inverted_img)
    img_np = np.where(img_np > 128, 255, 0)
    b_w = Image.fromarray(img_np.astype(np.uint8))
    text = pytesseract.image_to_string(b_w)
    score = int(text)
    return score

In [ ]:
target = 100
score = -1
colour = (0, 0, 0)
filename = "scores.csv"

if not os.path.exists(filename):
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Attempt', 'Score'])

counter = 0
if os.path.exists(filename):
    df = pd.read_csv(filename)
    if not df.empty:
        counter = df['Attempt'].max()

# Main game loop
while score < target:
    counter += 1
    
    # Start the game
    pyautogui.leftClick(start_button_pos[0], start_button_pos[1])
    time.sleep(59)
    
    # Wait for retry button to appear
    colour = pyautogui.pixel(retry_button_pos[0] * 2, retry_button_pos[1] * 2)
    tolerance = 10

    while not all(abs(c1 - c2) <= tolerance for c1, c2 in zip(colour, retry_button_colour)):
        colour = pyautogui.pixel(retry_button_pos[0] * 2, retry_button_pos[1] * 2)
        time.sleep(0.5)
    
    # Process successful attempt
    print(f"Attempt {counter}:")
    score = extract_score()
    
    # Save score to CSV
    with open(filename, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([counter, score])
    
    # Check if target reached
    if score >= target:
        print(f"Target value reached after {counter} attempts")
        break
    
    # Wait for Python interpreter to close
    check_intepreter_colour = pyautogui.pixel(python_interpreter[0] * 2, python_interpreter[1] * 2)
    while check_intepreter_colour == intepreter_colour:
        check_intepreter_colour = pyautogui.pixel(python_interpreter[0] * 2, python_interpreter[1] * 2)
        time.sleep(0.5)
    
    # Click retry button for next attempt
    pyautogui.leftClick(retry_button_pos[0], retry_button_pos[1])
    time.sleep(1)

KeyboardInterrupt: 